# El vaso no basta · contexto largo con la misma GRU

**Meta.** Misma máquina del lab 0, distinto $L\in\{32,64,128,256\}$. Mismas épocas, mismo LR, mismo ancho. Solo cambia cuántas letras ve de un jalón.

## Dummy

La hipótesis ingenua: “si le doy más letras atrás, predice mejor”. Suena lógico. En una GRU es **falso a menudo**.

Por qué: la GRU no recibe esas letras sueltas. Recibe un resumen $h$ (el vaso). Alargar $L$ de 64 a 256 le da **más tarea al vaso, no más vaso**. Además el gradiente tiene que viajar 256 pasos hacia atrás; a menudo se diluye y el modelo ni se entera de la letra lejana.

Vas a ver curvas que *no* bajan al subir $L$. Esa gráfica es el argumento para cambiar de máquina: **atención**. Aquí se acaba la GRU como eje del curso. Una sesión, no seis.

## Matemática

La información que el prefijo lejano puede meter en la predicción está acotada por la capacidad del vaso:

$$
I(x_{\le t-L};\,x_t\mid h_{t-1})\;\le\;I(h_{t-1};\,x_t),
$$

y $h_{t-1}\in\mathbb{R}^{d}$ tiene dimensión **fija**. Alargar $L$ no aumenta $d$. Si el gradiente $\partial\mathcal{L}/\partial h_{t-L}$ se desvanece, el vaso ni siquiera usa lo que sí cabría. Por eso “más contexto” no es monótono.

En cristiano: $I$ es “cuánto me entero de $A$ si veo $B$”. El pasado lejano solo puede influir *a través* del post-it $h$. Un post-it de 128 números no se vuelve más grande porque leas 256 letras.

## Cómo leer la tabla

Si $L=256$ sale *peor* que $L=64$, no rompiste PyTorch. Rompiste la hipótesis. Anótalo: es la frase de cierre del vaso.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/datos/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from pathlib import Path

# Local: ../datos/quijote.txt (o ../../ desde extra/). Colab: el wget lo deja junto al notebook.
TEXT_PATH = next(
    (str(p) for p in (Path("quijote.txt"), Path("../datos/quijote.txt"), Path("../../datos/quijote.txt")) if p.exists()),
    "quijote.txt",
)
print(f"device={DEVICE}  archivo={TEXT_PATH}")


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


In [ ]:
class CharGRULM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T) → logits (B, T, V) y vaso final hN
        emb = self.embed(x)
        out, hN = self.gru(emb, h0)
        logits = self.fc(out)
        return logits, hN


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


In [ ]:
@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens: int = 400, temperature: float = 1.0, device="cpu"):
    """Auto-regresivo: predice, pega, predice otra vez.

    El primer token nuevo sale de los logits del ÚLTIMO carácter del prompt.
    (Antes el código volvía a comer ese carácter: el texto salía corrido.)
    """
    model.eval()
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out_ids = ids.copy()
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    logits, h = model(x, h0=None)
    for _ in range(max_new_tokens):
        z = logits[0, -1, :] / max(1e-8, temperature)
        probs = torch.softmax(z, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        out_ids.append(next_id)
        inp = torch.tensor([[next_id]], dtype=torch.long, device=device)
        logits, h = model(inp, h0=h)
    return decode(out_ids, itos)


## Experimento

Cuatro $L$, dos épocas en CPU (sube si hay GPU). Misma semilla de ventanas. Al final: tabla $L$ vs val_loss / ppl, y **el mismo prompt** a cada modelo.

Entregable en una frase: ¿el vaso basta para un noticiero de 10 notas? (spoiler de producto: no. Las notas 1 y 10 no caben en $h$.)


In [ ]:
@dataclass
class ExpConfig:
    text_path: str = TEXT_PATH
    device: str = DEVICE
    batch_size: int = 32
    emb_dim: int = 128
    hidden_dim: int = 128
    num_layers: int = 1
    lr: float = 3e-4
    epochs: int = 2
    max_windows: int = 4000
    seq_lens: Tuple[int, ...] = (32, 64, 128, 256)
    prompt: str = "En un lugar de la Mancha "
    temperature: float = 1.0
    gen_tokens: int = 250


def run_experiment(cfg: ExpConfig):
    text = load_text(cfg.text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)
    n = len(data_ids)
    split = int(0.9 * n)
    train_ids, val_ids = data_ids[:split], data_ids[split:]
    device = torch.device(cfg.device)
    results = []

    print(f"Vocab={len(stoi)} | chars={n} | device={cfg.device}")
    for L in cfg.seq_lens:
        train_ds = CharWindowDataset(train_ids, L, max_windows=cfg.max_windows, seed=0)
        val_ds = CharWindowDataset(val_ids, L, max_windows=min(1500, cfg.max_windows), seed=1)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

        model = CharGRULM(len(stoi), cfg.emb_dim, cfg.hidden_dim, cfg.num_layers).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        print("\n" + "-" * 70)
        print(f"seq_len L={L} | train_windows={len(train_ds)} | val_windows={len(val_ds)}")
        tr = vl = float("nan")
        for epoch in range(1, cfg.epochs + 1):
            tr = train_one_epoch(model, train_loader, optimizer, device)
            vl = eval_loss(model, val_loader, device)
            ppl = math.exp(vl) if vl < 20 else float("inf")
            print(f"  Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

        sample = generate(model, cfg.prompt, stoi, itos, cfg.gen_tokens, cfg.temperature, str(device))
        ppl = math.exp(vl) if vl < 20 else float("inf")
        results.append((L, tr, vl, ppl, sample))
    return results


cfg = ExpConfig()
print("CPU a propósito: 2 épocas, modelo chico. GPU: hidden_dim=256, epochs=3.")
results = run_experiment(cfg)

print("\n" + "=" * 80)
print("RESUMEN (L vs val_loss / val_ppl)")
print("Hipótesis a romper: a mayor L, siempre mejora. A menudo NO.")
for L, tr, vl, ppl, _ in results:
    print(f"L={L:3d} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

print("\n" + "=" * 80)
print("MUESTRAS (una por L, misma temperatura)")
for L, _, _, _, sample in results:
    print("\n" + "-" * 70)
    print(f"L={L}")
    print(sample[:1200])
